## Import libraries


In [ ]:
import os
import pickle
import sys
import time

source_folder = "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/src"
sys.path.append(source_folder)

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm
from utils.utils import evaluate_and_save_outputs, load_config, save_config, set_seed

plt.rcParams["font.family"] = "DeJavu Serif"
plt.rcParams["font.serif"] = "Times New Roman"

import warnings

warnings.filterwarnings("ignore")

out_dir = "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/output/figures"

## Read the Germany shapefile


In [ ]:
# Read the NUTS1 and NUTS3 shapefile for DE
de_nuts1_gdf = gpd.read_file(
    os.path.join("/beegfs/halder/DATA/", "DE_NUTS", "DE_NUTS_3.shp")
)
de_nuts1_gdf = de_nuts1_gdf[
    de_nuts1_gdf["LEVL_CODE"] == 1
]  # filter only NUT1 level code

de_nuts3_gdf = gpd.read_file(
    os.path.join("/beegfs/halder/DATA/", "DE_NUTS", "DE_NUTS_3.shp")
)
de_nuts3_gdf = de_nuts3_gdf[
    de_nuts3_gdf["LEVL_CODE"] == 3
]  # filter only NUT3 level code

fig, ax = plt.subplots(figsize=(8, 8))
de_nuts3_gdf.plot(
    ax=ax,
    column="NUTS_NAME",
    cmap="Set3",
    edgecolor="grey",
    linewidth=0.5,
    label="NUTS3",
)
de_nuts1_gdf.plot(ax=ax, facecolor="none", edgecolor="k", linewidth=1, label="NUTS1")
plt.show()

print(de_nuts1_gdf.shape, de_nuts3_gdf.shape)
de_nuts3_gdf.head()

## Prepare the results


In [ ]:
# Function to prepare results
def prepare_results(model_name, crop, month, baseline=False):
    target_mean = cfg.scalers["yield_mean"]
    target_std = cfg.scalers["yield_std"]

    train_dataset = pd.read_parquet(
        f"/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/data/processed/{crop}/timeseries/DE11A_2004.parquet"
    )
    train_dataset["date"] = pd.to_datetime(train_dataset["date"]).dt.strftime("%b-%d")
    date_list = train_dataset["date"].tolist()[: model_config.get("seq_length")]

    if baseline != True:
        result_dir = os.path.join(
            f"/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/src/train/forecast/{crop}/{month}"
        )

    else:
        result_dir = os.path.join(
            f"/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/src/train/baseline/{crop}/{month}/{model_name}"
        )

    with open(os.path.join(result_dir, f"train_outputs.pkl"), "rb") as f:
        train_results = pickle.load(f)

    with open(os.path.join(result_dir, f"validation_outputs.pkl"), "rb") as f:
        val_results = pickle.load(f)

    with open(os.path.join(result_dir, f"test_outputs.pkl"), "rb") as f:
        test_results = pickle.load(f)

    combined_results = {
        k: np.concatenate([train_results[k], test_results[k], val_results[k]], axis=0)
        for k in test_results.keys()
    }

    # Prepare final predictions
    preds = combined_results["prediction"] * target_std + target_mean
    actuals = combined_results["target"] * target_std + target_mean
    prediction_df = pd.DataFrame(
        preds, columns=[f"pred_yield_q{i}" for i in model_config["quantiles"]]
    )
    prediction_df["target_yield"] = actuals
    prediction_df["year"] = combined_results["year"]
    prediction_df["NUTS_ID"] = combined_results["NUTS_ID"]
    prediction_df = prediction_df[
        ["NUTS_ID", "year", "target_yield"]
        + [f"pred_yield_q{i}" for i in model_config["quantiles"]]
    ]
    prediction_df.sort_values(by=["year", "NUTS_ID"], inplace=True)
    prediction_df.reset_index(drop=True, inplace=True)

    if baseline == False:
        keys_to_keep = {
            "NUTS_ID",
            "year",
            "static_weights",
            "temporal_weights",
            "attention_weights",
        }

        weights = {
            k: combined_results[k] for k in keys_to_keep if k in combined_results
        }

        # Prepare the static weights dataframe
        static_weights_df = pd.DataFrame(
            weights["static_weights"][:, :, 0], columns=cfg.static_real_variables
        )
        static_weights_df["NUTS_ID"] = weights["NUTS_ID"]
        static_weights_df["year"] = weights["year"]
        static_weights_df = static_weights_df[
            ["NUTS_ID", "year"] + cfg.static_real_variables
        ]
        static_weights_df.sort_values(by=["year", "NUTS_ID"], inplace=True)
        static_weights_df.reset_index(drop=True, inplace=True)

        # Prepare the temporal weights dataframe
        temporal_weights_array = weights["temporal_weights"][:, :, :, 0]
        n_samples, n_timesteps, n_vars = temporal_weights_array.shape

        temporal_weights_df = pd.DataFrame(
            temporal_weights_array.reshape(-1, n_vars), columns=cfg.time_varying_real
        )

        temporal_weights_df["NUTS_ID"] = np.repeat(weights["NUTS_ID"], n_timesteps)
        temporal_weights_df["year"] = np.repeat(weights["year"], n_timesteps)
        temporal_weights_df["timestep"] = np.tile(range(len(date_list)), n_samples)

        temporal_weights_df = temporal_weights_df[
            ["NUTS_ID", "year", "timestep"] + cfg.time_varying_real
        ]

        temporal_weights_df.sort_values(
            by=["year", "NUTS_ID", "timestep"], inplace=True
        )

        temporal_weights_df.reset_index(drop=True, inplace=True)

        result = (
            prediction_df,
            static_weights_df,
            temporal_weights_df,
            date_list,
        )

        return result

    else:
        return (
            prediction_df,
            date_list,
        )

In [ ]:
# Get results for all crops for CropFusionNet and all baselines
crops = ["winter_wheat", "winter_barley", "silage_maize"]

crop_labels = {
    "winter_barley": "Winter Barley",
    "winter_wheat": "Winter Wheat",
    "silage_maize": "Silage Maize",
}
# Forecast month per crop
crop_months = {
    "winter_barley": "Jul",
    "winter_wheat": "Jul",
    "silage_maize": "Sep",
}

baseline_models = ["VanillaLSTM", "SimpleTransformer", "ResCNN"]
baseline_labels = {
    "VanillaLSTM": "Vanilla LSTM",
    "SimpleTransformer": "Simple Transformer",
    "ResCNN": "ResCNN",
}

# Load and store all results in a dictionary keyed by crop name
all_crop_results = {}
for crop_name in crops:
    global cfg, model_config, train_config
    cfg, model_config, train_config = load_config(crop_name)
    month = crop_months[crop_name]

    # CropFusionNet (main model)
    prediction_df, static_weights_df, temporal_weights_df, date_list = prepare_results(
        "CropFusionNet", crop=crop_name, month=month, baseline=False
    )
    all_crop_results[crop_name] = {
        "CropFusionNet": {
            "prediction_df": prediction_df,
            "static_weights_df": static_weights_df,
            "temporal_weights_df": temporal_weights_df,
            "date_list": date_list,
        }
    }
    print(f"[{crop_labels[crop_name]}] CropFusionNet: {prediction_df.shape[0]} samples")

    # Baseline models
    for model_name in baseline_models:
        pred_df_bl, date_list_bl = prepare_results(
            model_name, crop=crop_name, month=month, baseline=True
        )
        all_crop_results[crop_name][model_name] = {
            "prediction_df": pred_df_bl,
            "date_list": date_list_bl,
        }
        print(
            f"[{crop_labels[crop_name]}] {baseline_labels[model_name]}: {pred_df_bl.shape[0]} samples"
        )

## Spatial distribution of CropFusionNet predictive error (rRMSE per NUTS3 region)


In [ ]:
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.stats import gaussian_kde


# ── helper: per-district metrics ─────────────────────────────────────────────
def compute_spatial_metrics(prediction_df, pred_col="pred_yield_q0.5"):
    """Compute per-NUTS3 rRMSE (%) and Pearson r across all years."""
    records = []
    for nuts_id, grp in prediction_df.groupby("NUTS_ID"):
        y_true = grp["target_yield"].values
        y_pred = grp[pred_col].values
        if len(y_true) < 2:
            continue
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        rrmse = rmse / np.mean(y_true) * 100
        r = np.corrcoef(y_true, y_pred)[0, 1]
        records.append({"NUTS_ID": nuts_id, "rRMSE": rrmse, "r": r})
    return pd.DataFrame(records)


# ── helper: KDE inset ─────────────────────────────────────────────────────────
def add_kde_inset(
    parent_ax,
    values,
    color,
    median_val,
    unit,
    loc="lower right",
    width="40%",  # Slightly wider to accommodate larger fonts
    height="28%",
    # (x, y, width, height) - nudging x to 1.05 moves it slightly right of the map
    bbox_to_anchor=(0.05, 0.05, 0.9, 0.95),
):
    axins = inset_axes(
        parent_ax,
        width=width,
        height=height,
        loc=loc,
        bbox_to_anchor=bbox_to_anchor,
        bbox_transform=parent_ax.transAxes,
        borderpad=0,  # Controlled by bbox_to_anchor now
    )

    kde = gaussian_kde(values, bw_method=0.4)
    xs = np.linspace(values.min(), values.max(), 300)
    ys = kde(xs)

    axins.fill_between(xs, ys, alpha=0.35, color=color)
    axins.plot(xs, ys, color=color, lw=1.5)

    # Rug marks
    axins.plot(
        values,
        np.full_like(values, -0.02 * ys.max()),
        "|",
        color=color,
        alpha=0.4,
        markersize=4,
    )

    # Median line and Text - INCREASED FONT SIZE
    axins.axvline(median_val, color="black", lw=1.5, ls="--")
    axins.text(
        median_val,
        ys.max() * 1.15,
        f"{median_val:.1f}{unit}",
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold",
        color="black",
    )

    axins.set_yticks([])
    # Tick labels - INCREASED FONT SIZE
    axins.tick_params(axis="x", labelsize=14, pad=2)

    axins.set_xlim(
        values.min() - 0.05 * np.ptp(values), values.max() + 0.05 * np.ptp(values)
    )
    axins.set_ylim(-0.06 * ys.max(), ys.max() * 1.1)

    axins.set_facecolor("none")
    axins.patch.set_alpha(0)
    for spine in axins.spines.values():
        spine.set_visible(False)

    return axins


# ── pre-compute metrics ───────────────────────────────────────────────────────
crop_order = ["winter_wheat", "winter_barley", "silage_maize"]
panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]

metrics_per_crop = {}
for crop_name in crop_order:
    pred_df = all_crop_results[crop_name]["CropFusionNet"]["prediction_df"]
    pred_df = pred_df[pred_df["year"] >= 2019]
    metrics_per_crop[crop_name] = compute_spatial_metrics(pred_df)

# shared colour ranges (2nd–98th percentile across all crops)
all_rrmse = [v for c in crop_order for v in metrics_per_crop[c]["rRMSE"].tolist()]
all_r = [v for c in crop_order for v in metrics_per_crop[c]["r"].tolist()]
rrmse_vmin, rrmse_vmax = np.percentile(all_rrmse, 2), np.percentile(all_rrmse, 98)
r_vmin, r_vmax = np.percentile(all_r, 2), np.percentile(all_r, 98)

# KDE accent colours (one per row)
kde_colors = {"rRMSE": "#d62728", "r": "#2ca02c"}

# ── figure: 2 rows × 3 cols ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 14))

row_configs = [
    (0, "rRMSE", "RdYlGn_r", rrmse_vmin, rrmse_vmax, "rRMSE (%)"),
    (1, "r", "RdYlGn", r_vmin, r_vmax, "Pearson $r$"),
]

for row_idx, metric_col, cmap, vmin, vmax, cbar_label in row_configs:
    panel_offset = row_idx * 3
    unit = "%" if metric_col == "rRMSE" else ""

    for col_idx, crop_name in enumerate(crop_order):
        ax = axes[row_idx, col_idx]
        metrics_df = metrics_per_crop[crop_name]
        merged = de_nuts3_gdf.merge(metrics_df, on="NUTS_ID", how="left")
        panel = panel_labels[panel_offset + col_idx]

        # grey background
        de_nuts3_gdf.plot(ax=ax, color="#d9d9d9", edgecolor="grey", linewidth=0.3)

        # choropleth
        merged.plot(
            ax=ax,
            column=metric_col,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            edgecolor="grey",
            linewidth=0.3,
            missing_kwds={"color": "#d9d9d9"},
        )

        # NUTS1 state borders
        de_nuts1_gdf.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1.2)

        median_val = metrics_df[metric_col].median()

        ax.set_title(
            f"{panel} {crop_labels[crop_name]}",
            fontsize=18,
            fontweight="bold",
            pad=6,
        )
        ax.set_axis_off()

        # ── KDE inset (lower-right corner) ───────────────────────────────────
        add_kde_inset(
            ax,
            values=metrics_df[metric_col].dropna().values,
            color=kde_colors[metric_col],
            median_val=median_val,
            unit=unit,
            loc="lower right",
            width="38%",
            height="26%",
            bbox_to_anchor=(0, 0, 1.35, 1),
        )

    if row_idx == 0:
        cax = fig.add_axes([0.975, 0.67, 0.015, 0.28])
    else:
        cax = fig.add_axes([0.975, 0.15, 0.015, 0.28])

    # shared colourbar per row
    sm = ScalarMappable(cmap=cmap, norm=Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(
        sm,
        cax=cax,
        orientation="vertical",
    )
    cbar.set_label(cbar_label, fontsize=16)
    cbar.ax.tick_params(labelsize=14)

# row labels on the left margin
for row_idx, title in enumerate(["rRMSE", "Pearson $r$"]):
    axes[row_idx, 0].annotate(
        title,
        xy=(-0.04, 0.5),
        xycoords="axes fraction",
        fontsize=16,
        fontweight="bold",
        ha="right",
        va="center",
        rotation=90,
    )

plt.tight_layout()
plt.subplots_adjust(hspace=0.15)
plt.savefig(
    os.path.join(out_dir, "final", "spatial_error_distribution_CropFusionNet.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()
print("Figure saved.")